#### Seq2Seq Model Basics
- It is divided into 3 parts
1) the Encoder
    - the basic job is to read input sequence, compute is compress it into single dence vector
    - It uses RNN/LSTM/GRU
    - The output will be a final hidden state h_t also called content vector (c)
    - eg: Convert `Hello world` the `Hola mundo`
        - The encoder converts Hello world to its context vector c
2) context vector
    - The concept of contest vector is to store the thought or concepts read by the Encoder.
    - It holds the whole meaning of the content read 
    - Because it is a fixed size, It has to heavily squeeze potential long sentences into the exact amount space of a short sentence this is called infomation bottleneck
3) The Decoder
   - The basic concept is to take the context vector from the ecnoder as input and give the output
   - Explination :-
        - The first hidden state of decoder($s_0$) is set of the context vector (c)
        - Then we feed `<sos>` token to tell decoder to start the process
        - The decoder looks at the input and the current hidden i.e the context vector, and predicts the output
            - The output is predicted as `hola`
        - It also creates a new hidden state ($s_1$) the is used to predict the next word
        - The next stage, the output of the first stage and its current hidden state is taken and output is predicted
            - $S_1$ and `hola` is taken as input and `mundo` is given as output
        - this process repeats till the final token `<eos>` is given. This tells that the process is complete
4) Teacher Forcing
    - The major drawback of the decoder is that when it predicts incorrectly at first hidden stage and that incorrect output is given as input to hidden stage 2, then while finding loss and backpropogation, the model will find it difficult to learn about second output as it thinks the first output is correct. 
    - to solve this `Teacher Forcing` is used. When the output of first hidden state is wong, we given the correct output from the context vector as input to the second hidden state. This helps the model to learn faster and perform better.
    - If we use this method all the time, the model might become completely dependent of the previous output being right all the time. Hence, we use a `Teacher Forcing Ratio`. For every word generation step during training, we flip a coin: 50% of the time we give it the true previous word (Teacher Forcing), and 50% of the time we make it use its own previous prediction (Autoregression). This builds robustness while still speeding up training.

In [86]:
import random
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split

# -----------------------------
# 1. Build vocabulary
# -----------------------------
special_tokens = ["<pad>", "<sos>", "<eos>"]
digit_tokens = [str(i) for i in range(10)]

vocab = special_tokens + digit_tokens
stoi = {token: idx for idx, token in enumerate(vocab)}
itos = {idx: token for token, idx in stoi.items()}

PAD_IDX = stoi["<pad>"]
SOS_IDX = stoi["<sos>"]
EOS_IDX = stoi["<eos>"]

print("Vocabulary:", stoi)


# -----------------------------
# 2. Dataset for reverse-sequence task
# -----------------------------
class ReverseSequenceDataset(Dataset):
    def __init__(self, num_samples=1000, min_len=3, max_len=7):
        self.samples = []

        for _ in range(num_samples):
            seq_len = random.randint(min_len, max_len)

            # Random digits from 0 to 9 converted to token IDs
            src = [stoi[str(random.randint(0, 9))] for _ in range(seq_len)]

            # Target = <sos> + reversed(src) + <eos>
            trg = [SOS_IDX] + src[::-1] + [EOS_IDX]

            self.samples.append((src, trg))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        src, trg = self.samples[idx]
        return torch.tensor(src, dtype=torch.long), torch.tensor(trg, dtype=torch.long)


# -----------------------------
# 3. Collate function for padding
# -----------------------------
def collate_fn(batch):
    src_batch, trg_batch = zip(*batch)

    src_lengths = [len(seq) for seq in src_batch]
    trg_lengths = [len(seq) for seq in trg_batch]

    max_src_len = max(src_lengths)
    max_trg_len = max(trg_lengths)

    padded_src = []
    padded_trg = []

    for src, trg in zip(src_batch, trg_batch):
        src_pad = torch.cat([
            src,
            torch.full((max_src_len - len(src),), PAD_IDX, dtype=torch.long)
        ])
        trg_pad = torch.cat([
            trg,
            torch.full((max_trg_len - len(trg),), PAD_IDX, dtype=torch.long)
        ])

        padded_src.append(src_pad)
        padded_trg.append(trg_pad)

    padded_src = torch.stack(padded_src)   # shape: [batch_size, max_src_len]
    padded_trg = torch.stack(padded_trg)   # shape: [batch_size, max_trg_len]

    return padded_src, padded_trg


# -----------------------------
# 4. Create dataset
# -----------------------------
dataset = ReverseSequenceDataset(num_samples=50000, min_len=3, max_len=7)

# Train/validation split
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# DataLoaders
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn
)


# -----------------------------
# 5. Check one batch
# -----------------------------
def decode(sequence):
    return [itos[idx] for idx in sequence]

src_batch, trg_batch = next(iter(train_loader))

print("\nSource batch shape:", src_batch.shape)
print("Target batch shape:", trg_batch.shape)

print("\nExample padded source:")
print(src_batch[0])
print("Decoded:", decode(src_batch[0].tolist()))

print("\nExample padded target:")
print(trg_batch[0])
print("Decoded:", decode(trg_batch[0].tolist()))

Vocabulary: {'<pad>': 0, '<sos>': 1, '<eos>': 2, '0': 3, '1': 4, '2': 5, '3': 6, '4': 7, '5': 8, '6': 9, '7': 10, '8': 11, '9': 12}

Source batch shape: torch.Size([32, 7])
Target batch shape: torch.Size([32, 9])

Example padded source:
tensor([12,  6, 11,  4, 10, 10, 11])
Decoded: ['9', '3', '8', '1', '7', '7', '8']

Example padded target:
tensor([ 1, 11, 10, 10,  4, 11,  6, 12,  2])
Decoded: ['<sos>', '8', '7', '7', '1', '8', '3', '9', '<eos>']


In [106]:
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, embed_dim, dropout_p=0.5):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, embed_dim)
        self.dropout = nn.Dropout(dropout_p)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
    def forward(self, src):
        embedded = self.embedding(src)  # [batch_size, src_len, embed_dim]
        embedded = self.dropout(embedded)
        out, hidden = self.gru(embedded) # out: [batch_size, src_len, hidden_dim], hidden: [1, batch_size, hidden_dim]
        return out, hidden

In [107]:
class Decoder(nn.Module):
    def __init__(self, output_dim, hidden_dim, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, input_token, hidden):
        input_token = input_token.unsqueeze(1)  # [batch_size, 1]
        embedded = self.embedding(input_token)  # [batch_size, 1, embed_dim]
        out, hidden = self.gru(embedded, hidden)  # out: [batch_size, 1, hidden_dim]
        prediction = self.fc(out.squeeze(1))  # [batch_size, output_dim]
        return prediction, hidden

In [108]:
class Seq2Seq(nn.Module):
    def __init__(self, encode, decode, device):
        super().__init__()
        self.encoder = encode
        self.decoder = decode
        self.device = device
    
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.size(0)
        trg_len = trg.size(1)
        vocab_size = self.decoder.fc.out_features

        outputs = torch.zeros(batch_size, trg_len, vocab_size).to(self.device)
        encoder_out, hidden = self.encoder(src)
        input_token = trg[:, 0]  # Start with <sos>
        for t in range(1, trg_len):
            output, hidden = self.decoder(input_token, hidden)
            outputs[:, t, :] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input_token = trg[:, t] if teacher_force else top1  # Teacher forcing or greedy decoding
        return outputs

In [109]:
# Hyperparameters
vocab_size = len(stoi)
INPUT_DIM =  vocab_size # Size of source vocabulary
OUTPUT_DIM =  vocab_size # Size of target vocabulary
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HIDDEN_DIM = 128
num_epochs = 20
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Init models
enc = Encoder(INPUT_DIM, HIDDEN_DIM, ENC_EMB_DIM)
dec = Decoder(OUTPUT_DIM, HIDDEN_DIM, DEC_EMB_DIM)
model = Seq2Seq(enc, dec, device).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = torch.optim.Adam(model.parameters(), lr = 0.0001, weight_decay=1e-5)

for epoch in range(num_epochs):
    model.train()
    running_train_loss = 0.0
    train_correct = 0
    train_total = 0

    for src, trg in train_loader:
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()

        outputs = model(src, trg)                 
        output_dim = outputs.shape[-1]

        outputs = outputs[:, 1:, :].reshape(-1, output_dim)
        targets = trg[:, 1:].reshape(-1)

        loss = criterion(outputs, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_train_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)
        non_pad_mask = targets != PAD_IDX
        train_correct += (preds[non_pad_mask] == targets[non_pad_mask]).sum().item()
        train_total += non_pad_mask.sum().item()

    epoch_train_loss = running_train_loss / len(train_loader)
    epoch_train_acc = train_correct / train_total


    # -------------------------
    # Validation
    # -------------------------
    model.eval()
    running_val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for src, trg in val_loader:
            src, trg = src.to(device), trg.to(device)

            outputs = model(src, trg)   # same forward pass as training
            output_dim = outputs.shape[-1]

            outputs_flat = outputs[:, 1:, :].reshape(-1, output_dim)
            targets = trg[:, 1:].reshape(-1)

            loss = criterion(outputs_flat, targets)
            running_val_loss += loss.item()

            preds = torch.argmax(outputs_flat, dim=1)

            non_pad_mask = targets != PAD_IDX
            val_correct += (preds[non_pad_mask] == targets[non_pad_mask]).sum().item()
            val_total += non_pad_mask.sum().item()

    epoch_val_loss = running_val_loss / len(val_loader)
    epoch_val_acc = val_correct / val_total

    print(
        f"Epoch [{epoch+1}/{num_epochs}] | "
        f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}"
    )


Epoch [1/20] | Train Loss: 1.4430 | Train Acc: 0.5340 | Val Loss: 0.8047 | Val Acc: 0.7478
Epoch [2/20] | Train Loss: 0.6861 | Train Acc: 0.7604 | Val Loss: 0.5489 | Val Acc: 0.8010
Epoch [3/20] | Train Loss: 0.5033 | Train Acc: 0.8186 | Val Loss: 0.4275 | Val Acc: 0.8411
Epoch [4/20] | Train Loss: 0.4074 | Train Acc: 0.8538 | Val Loss: 0.3541 | Val Acc: 0.8678
Epoch [5/20] | Train Loss: 0.3394 | Train Acc: 0.8794 | Val Loss: 0.2866 | Val Acc: 0.8936
Epoch [6/20] | Train Loss: 0.2789 | Train Acc: 0.9046 | Val Loss: 0.2297 | Val Acc: 0.9175
Epoch [7/20] | Train Loss: 0.2353 | Train Acc: 0.9221 | Val Loss: 0.1913 | Val Acc: 0.9350
Epoch [8/20] | Train Loss: 0.1967 | Train Acc: 0.9372 | Val Loss: 0.1580 | Val Acc: 0.9481
Epoch [9/20] | Train Loss: 0.1704 | Train Acc: 0.9479 | Val Loss: 0.1367 | Val Acc: 0.9554
Epoch [10/20] | Train Loss: 0.1525 | Train Acc: 0.9551 | Val Loss: 0.1186 | Val Acc: 0.9633
Epoch [11/20] | Train Loss: 0.1314 | Train Acc: 0.9625 | Val Loss: 0.1037 | Val Acc: 0.96

In [113]:
itos = {idx: tok for tok, idx in stoi.items()}

def predict_sequence(model, seq, max_len=None):
    model.eval()

    seq = [str(x) for x in seq]
    src = torch.tensor([stoi[x] for x in seq], dtype=torch.long).unsqueeze(0).to(device)

    if max_len is None:
        max_len = len(seq) # Add space for <sos> and <eos>

    with torch.no_grad():
        _, hidden = model.encoder(src)

        inp = torch.full((1,), stoi['<sos>'], dtype=torch.long, device=device)
        result = []

        for _ in range(max_len):
            output, hidden = model.decoder(inp, hidden)
            pred = output.argmax(dim=1).item()

            if pred == stoi['<eos>']:
                break

            result.append(itos[pred])
            inp.fill_(pred)

    return result

print(predict_sequence(model, [4, 5, 6]))


['6', '5', '5']


#### Context vector bottleneck and Attention Mechanism

1) Context Vector Bottleneck
- The main problem with current Seq2Seq model is that during encoder phase, a final hidden layer provides a context vector. I contains all the informations that is acquired from all previous hidden layers.
- A context vector has a limited space and if the data is less(like a few sentence), then this works. But, if the data is large (like paragraphs), Then, by the time the encoder reaches the final hidden layer, it forgets the informations from the initial hidden layers. 

2) Attention
- To solve this bottlenech, Attention mechanism is used. 
- Insted of compressing all the information in one context vector, the attention mechanism tell the encoder to keep all the hidden states

3) Attention Mechanism
- Lets say that the decoder has predicted one hidden state ($s_t$). It take the current hidden state and compares it with all the hidden states from the encoder
    - While comparing ($s_t$) with encoder hidden states ($h_t$), It asks: "How mathematically similar is my current thought ($s_t$) to the encoder's memory of the word 'The' ($h_1$)?"
    - To find the similarity, it uses **The dot product(The geometric Way)** or **The Matchmaker Network (The Additive Way)**
    - We use **The Additive way also know as Bahdanau Attention**. 
                    $$e_{t,i} = v^T \tanh(W [s_t; h_i])$$
        - Concatenation ($[s_t; h_i]$): We literally glue the decoder's thought vector and the encoder's memory vector together into one giant vector.
        - The Linear Layer ($W$): We pass this giant glued vector through a Linear layer. This layer mixes the numbers together, allowing the network to find complex relationships between the thought and the memory.
        - Activation ($\tanh$): We pass the result through a tanh function to keep the numbers stabilized between -1 and 1.
        - The Final Score ($v^T$): We multiply the result by a final set of learned weights ($v$) to compress it all down into a single, raw number—the alignment score.
    - The similarity score get better as the model gets trained better
- These similarity score are them passed into a softmax function to turn into a percentage, where the additive of all will be 1
- We take the encoder's memory of the ($h_5$), multiply it by its corresponding softmax percentage, do the same for the rest of the words with their tiny percentages, and add them all together.
                $c_t = \sum_{i=1}^{T} \alpha_{t,i} h_i$
    - (Where $c_t$ is the new context vector for step $t$, $\alpha$ is the attention weight, and $h$ is the encoder hidden state).
- Finally, the decoder takes this custom context vector, combines it with its own current hidden state, and passes it through the Linear layer to predict the word
- Then, it moves to the next step, calculates brand new attention weights, builds a brand new context vector, and predicts the next word.

In [95]:
class AttentionEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, embed_dim, dropout_p = 0.5):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, embed_dim)
        self.dropout = nn.Dropout(dropout_p)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
    def forward(self, src):
        embedded = self.embedding(src)  # [batch_size, src_len, embed_dim]
        embedded = self.dropout(embedded)
        out, hidden = self.gru(embedded) # out: [batch_size, src_len, hidden_dim], hidden: [1, batch_size, hidden_dim]
        return out, hidden

In [96]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim + hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        # hidden: [batch_size, hidden_dim]
        # encoder_outputs: [batch_size, src_len, hidden_dim]

        src_len = encoder_outputs.shape[1]

        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)
        # [batch_size, src_len, hidden_dim]

        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        # [batch_size, src_len, hidden_dim]

        attention = self.v(energy).squeeze(2)
        # [batch_size, src_len]

        return torch.softmax(attention, dim=1)

In [97]:
class AttentionDecoder(nn.Module):
    def __init__(self, output_dim, hidden_dim, embed_dim, attention):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, embed_dim)
        self.attention = attention
        self.gru = nn.GRU(embed_dim + hidden_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, input_token, hidden, encoder_outputs):
        input_token = input_token.unsqueeze(1)   # [batch_size, 1]
        embedded = self.embedding(input_token)   # [batch_size, 1, embed_dim]

        attn_weights = self.attention(hidden[0], encoder_outputs)
        # [batch_size, src_len]

        attn_weights = attn_weights.unsqueeze(1)
        # [batch_size, 1, src_len]

        context = torch.bmm(attn_weights, encoder_outputs)
        # [batch_size, 1, hidden_dim]

        gru_input = torch.cat((embedded, context), dim=2)
        # [batch_size, 1, embed_dim + hidden_dim]

        out, hidden = self.gru(gru_input, hidden)
        # out: [batch_size, 1, hidden_dim]

        prediction = self.fc(out.squeeze(1))
        # [batch_size, output_dim]

        return prediction, hidden


In [98]:
class AttentionSeq2Seq(nn.Module):
    def __init__(self, encode, decode, device):
        super().__init__()
        self.encoder = encode
        self.decoder = decode
        self.device = device
    
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.size(0)
        trg_len = trg.size(1)
        vocab_size = self.decoder.fc.out_features

        outputs = torch.zeros(batch_size, trg_len, vocab_size).to(self.device)
        encoder_out, hidden = self.encoder(src)
        input_token = trg[:, 0]  # Start with <sos>
        for t in range(1, trg_len):
            output, hidden = self.decoder(input_token, hidden, encoder_out)
            outputs[:, t, :] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input_token = trg[:, t] if teacher_force else top1  # Teacher forcing or greedy decoding
        return outputs

In [101]:
# Hyperparameters
vocab_size = len(stoi)
INPUT_DIM =  vocab_size # Size of source vocabulary
OUTPUT_DIM =  vocab_size # Size of target vocabulary
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HIDDEN_DIM = 128
num_epochs = 20
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Init models
enc = AttentionEncoder(INPUT_DIM, HIDDEN_DIM, ENC_EMB_DIM)
attention = Attention(HIDDEN_DIM)
dec = AttentionDecoder(OUTPUT_DIM, HIDDEN_DIM, DEC_EMB_DIM, attention)
model = AttentionSeq2Seq(enc, dec, device).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = torch.optim.Adam(model.parameters(), lr = 0.0001, weight_decay=1e-5)

for epoch in range(num_epochs):
    model.train()
    running_train_loss = 0.0
    train_correct = 0
    train_total = 0

    for src, trg in train_loader:
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()

        outputs = model(src, trg)                 
        output_dim = outputs.shape[-1]

        outputs = outputs[:, 1:, :].reshape(-1, output_dim)
        targets = trg[:, 1:].reshape(-1)

        loss = criterion(outputs, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_train_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)
        non_pad_mask = targets != PAD_IDX
        train_correct += (preds[non_pad_mask] == targets[non_pad_mask]).sum().item()
        train_total += non_pad_mask.sum().item()

    epoch_train_loss = running_train_loss / len(train_loader)
    epoch_train_acc = train_correct / train_total


    # -------------------------
    # Validation
    # -------------------------
    model.eval()
    running_val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for src, trg in val_loader:
            src, trg = src.to(device), trg.to(device)

            outputs = model(src, trg)   # same forward pass as training
            output_dim = outputs.shape[-1]

            outputs_flat = outputs[:, 1:, :].reshape(-1, output_dim)
            targets = trg[:, 1:].reshape(-1)

            loss = criterion(outputs_flat, targets)
            running_val_loss += loss.item()

            preds = torch.argmax(outputs_flat, dim=1)

            non_pad_mask = targets != PAD_IDX
            val_correct += (preds[non_pad_mask] == targets[non_pad_mask]).sum().item()
            val_total += non_pad_mask.sum().item()

    epoch_val_loss = running_val_loss / len(val_loader)
    epoch_val_acc = val_correct / val_total

    print(
        f"Epoch [{epoch+1}/{num_epochs}] | "
        f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}"
    )


Epoch [1/20] | Train Loss: 1.0364 | Train Acc: 0.6744 | Val Loss: 0.1925 | Val Acc: 0.9611
Epoch [2/20] | Train Loss: 0.0660 | Train Acc: 0.9919 | Val Loss: 0.0301 | Val Acc: 0.9937
Epoch [3/20] | Train Loss: 0.0131 | Train Acc: 0.9987 | Val Loss: 0.0177 | Val Acc: 0.9959
Epoch [4/20] | Train Loss: 0.0065 | Train Acc: 0.9991 | Val Loss: 0.0118 | Val Acc: 0.9970
Epoch [5/20] | Train Loss: 0.0033 | Train Acc: 0.9995 | Val Loss: 0.0062 | Val Acc: 0.9981
Epoch [6/20] | Train Loss: 0.0036 | Train Acc: 0.9994 | Val Loss: 0.0023 | Val Acc: 0.9995
Epoch [7/20] | Train Loss: 0.0039 | Train Acc: 0.9993 | Val Loss: 0.0050 | Val Acc: 0.9986
Epoch [8/20] | Train Loss: 0.0029 | Train Acc: 0.9995 | Val Loss: 0.0094 | Val Acc: 0.9975
Epoch [9/20] | Train Loss: 0.0063 | Train Acc: 0.9990 | Val Loss: 0.0025 | Val Acc: 0.9995
Epoch [10/20] | Train Loss: 0.0008 | Train Acc: 0.9999 | Val Loss: 0.0205 | Val Acc: 0.9947
Epoch [11/20] | Train Loss: 0.0011 | Train Acc: 0.9998 | Val Loss: 0.0022 | Val Acc: 0.99

In [104]:
itos = {idx: tok for tok, idx in stoi.items()}

def predict_sequence(model, seq, max_len=None):
    model.eval()

    seq = [str(x) for x in seq]
    src = torch.tensor([stoi[x] for x in seq], dtype=torch.long).unsqueeze(0).to(device)

    if max_len is None:
        max_len = len(seq)

    with torch.no_grad():
        encoder_outputs, hidden = model.encoder(src)

        inp = torch.full((1,), stoi['<sos>'], dtype=torch.long, device=device)
        result = []

        for _ in range(max_len):
            output, hidden = model.decoder(inp, hidden, encoder_outputs)
            pred = output.argmax(dim=1).item()

            if pred == stoi['<eos>']:
                break

            result.append(itos[pred])
            inp.fill_(pred)

    return result

print(predict_sequence(model, [1, 2, 3]))


['3', '3', '3']
